In [ ]:
import pandas as pd
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import tensorflow.keras as keras
!pip install numpy==1.24.3 --quiet
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Embedding, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
names = ["class", "message"]

In [ ]:
train_file = pd.read_csv(train_file_path, sep='\t', names=names)
train_file

In [ ]:
test_file = pd.read_csv(test_file_path, sep='\t', names=names)
test_file

In [ ]:
train_message = train_file["message"].values.tolist()
train_label = np.array([0 if x=="ham" else 1 for x in train_file['class'].values.tolist()])
test_message = test_file["message"].values.tolist()
test_label = np.array([0 if x=="ham" else 1 for x in test_file['class'].values.tolist()])

In [ ]:
vocabulary = {}
for message in train_message:
  for vocab in message.split():
    if vocab not in vocabulary:
      vocabulary[vocab] = 1
    else:
      vocabulary[vocab] += 1

In [ ]:
VOCAB_SIZE = len(vocabulary)

longest_message = max(train_message, key=lambda p: len(p.split()))
longest_message_words = longest_message.split()
MAX_LENGTH = len(longest_message_words)

In [ ]:
encoded_train_message = [one_hot(x, VOCAB_SIZE) for x in train_message]
padded_train_message = pad_sequences(encoded_train_message, maxlen=MAX_LENGTH, padding='post')
encoded_test_message = [one_hot(x, VOCAB_SIZE) for x in test_message]
padded_test_message = pad_sequences(encoded_test_message, maxlen=MAX_LENGTH, padding='post')

In [ ]:
#Building the model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=100, input_length=MAX_LENGTH),
    Flatten(),
    Dense(1, activation='sigmoid')
])

#Compiling the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#Early Stopping callback
early_stop = EarlyStopping(
    monitor='val_accuracy',
    min_delta=1e-4,
    patience=25,
    verbose=1,
    mode='max',
    restore_best_weights=True
)

#Training the model
model.fit(
    padded_train_message,
    train_label,
    validation_data=(padded_test_message, test_label),
    callbacks=[early_stop],
    epochs=1000,
    verbose=2
)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  class_dict = {
      0: "ham",
      1: "spam",
  }
  encoded_message = [one_hot(pred_text, VOCAB_SIZE)]
  padded_message = pad_sequences(encoded_message, maxlen=MAX_LENGTH, padding="post")
  prediction = [model.predict(padded_message)[0][0], class_dict[np.round(model.predict(padded_message)[0][0])]]
  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
